# Adding `missing_values` counts to the enriched codebooks

Goal: for each column already described in `raw_data/hh09dta_b2/codebook_json/{table}_enriched.json`,
add a `"missing_values"` key holding the number of missing values for that column in the
corresponding `.dta` table.

The **original** `{table}_enriched.json` files are never modified. Each table gets a new file next
to it: `{table}_enriched_with_missing_values.json`.

## What counts as "missing" here

Checked what's actually in this dataset before deciding:

- No column has empty-string/whitespace-only values (only `folio`/`ls` are string-typed, and both
  are always populated) -- so there's no "empty space" case to handle for this dataset.
- `pandas.read_stata(...)` already converts Stata's system-missing (`.`) and extended missing
  (`.a`-`.z`) codes to `NaN`, so plain `.isna()` already covers *true* missingness.
- BUT several columns also carry a **documented non-response code** that is a real recorded value,
  not a `NaN` -- specifically a `"DK"` ("Don't Know") or `"NA"` ("Not Applicable", e.g. "were you
  robbed at your business" when you don't have one) label in that column's `values_label`. Those
  are non-answers just as much as a `NaN` is, but a raw `.isna()` count misses them entirely (e.g.
  `vlh16a` in `ii_vlh` has 1378 true `NaN` but 4313 more rows coded `"NA"` -- more than 3x as much
  real missingness as `.isna()` alone would report).

So `"missing_values"` below = `NaN` count **plus** the count of values matching a `"DK"` or `"NA"`
label for that column (if the column has one). No other placeholder values were found in this
dataset, so nothing else is added -- if you spot another one, this is easy to extend (see Step 1).

Run the cells top to bottom. Steps 1-3 build and sanity-check the logic on a single table (`ii_ah`,
matching the example you gave). Step 4 is optional and applies the same logic to every table in
`codebook_json/`.

## Step 0: paths

In [5]:
import json
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
DATA_DIR = ROOT / "raw_data" / "hh09dta_b2"
JSON_DIR = DATA_DIR / "codebook_json"

assert DATA_DIR.exists(), f"Missing: {DATA_DIR}"
assert JSON_DIR.exists(), f"Missing: {JSON_DIR}"

# labels treated as a non-response code rather than a real answer -- extend this
# set if you spot another placeholder label in the codebook (e.g. "Refused")
MISSING_LIKE_LABELS = {"DK", "NA"}


## Step 1: the enrichment function

Loads `{table}.dta` and `{table}_enriched.json`, and returns a **new dict** (the original JSON file
on disk is not touched) with `"missing_values"` added to every column that already has an entry in
the codebook.

`"missing_values"` = NaN count + count of values matching a `DK`/`NA`-labeled code for that column.
The breakdown (nan vs. coded) is printed per column so you can see the composition, even though only
the combined total is written to the JSON.

Columns present in the codebook but missing from the dataframe, and columns present in the
dataframe but undocumented in the codebook, are both reported (not silently dropped) so you can see
the gap -- e.g. `ii_ah.dta` has 65 columns but `ii_ah_enriched.json` only documents 47 of them.

In [6]:
def count_missing(df: pd.DataFrame, column_name: str, metadata: dict) -> tuple[int, int]:
    """Returns (nan_count, dk_or_na_coded_count) for one column."""

    nan_count = int(df[column_name].isna().sum())

    coded_count = 0
    values_label = metadata.get("values_label") or {}

    for label, code in values_label.items():
        if label not in MISSING_LIKE_LABELS:
            continue

        try:
            code_value = float(code)
        except (TypeError, ValueError):
            continue

        coded_count += int((df[column_name] == code_value).sum())

    return nan_count, coded_count


def add_missing_value_counts(table_name: str, verbose: bool = True) -> dict:
    dta_path = DATA_DIR / f"{table_name}.dta"
    enriched_path = JSON_DIR / f"{table_name}_enriched.json"

    df = pd.read_stata(dta_path)

    with open(enriched_path, "r", encoding="utf-8") as f:
        json_dict = json.load(f)

    skipped_columns = []

    for column_name, metadata in json_dict.items():
        if column_name not in df.columns:
            skipped_columns.append(column_name)
            continue

        nan_count, coded_count = count_missing(df, column_name, metadata)
        metadata["missing_values"] = str(nan_count + coded_count)

        if verbose and coded_count:
            print(
                f"[{table_name}] {column_name}: nan={nan_count} "
                f"dk_or_na_coded={coded_count} total={nan_count + coded_count}"
            )

    undocumented_columns = sorted(set(df.columns) - set(json_dict.keys()))

    if skipped_columns:
        print(f"[{table_name}] in codebook but not in dataframe (left as-is): {skipped_columns}")

    if undocumented_columns:
        print(f"[{table_name}] in dataframe but not in codebook (no missing_values added): {undocumented_columns}")

    return json_dict


## Step 2: try it on `ii_ah` and inspect the result before writing anything

In [7]:
table_name = "ii_ah"

enriched_with_missing = add_missing_value_counts(table_name)

# matches the example from the conversation: "ah03a" should now carry "missing_values"
enriched_with_missing["ah03a"]


[ii_ah] ah04a_1: nan=2001 dk_or_na_coded=1898 total=3899
[ii_ah] ah04b_1: nan=7287 dk_or_na_coded=457 total=7744
[ii_ah] ah04c_1: nan=5582 dk_or_na_coded=266 total=5848
[ii_ah] ah04d_1: nan=8119 dk_or_na_coded=98 total=8217
[ii_ah] ah04e_1: nan=707 dk_or_na_coded=846 total=1553
[ii_ah] ah04f_1: nan=1061 dk_or_na_coded=804 total=1865
[ii_ah] ah04g_1: nan=1383 dk_or_na_coded=775 total=2158
[ii_ah] ah04h_1: nan=7956 dk_or_na_coded=494 total=8450
[ii_ah] ah04i_1: nan=8906 dk_or_na_coded=43 total=8949
[ii_ah] ah04j_1: nan=8697 dk_or_na_coded=53 total=8750
[ii_ah] ah04k_1: nan=8641 dk_or_na_coded=52 total=8693
[ii_ah] ah04l_1: nan=8617 dk_or_na_coded=46 total=8663
[ii_ah] ah04m_1: nan=8035 dk_or_na_coded=102 total=8137
[ii_ah] ah04n_1: nan=9046 dk_or_na_coded=7 total=9053
[ii_ah] in dataframe but not in codebook (no missing_values added): ['ah05a', 'ah05b', 'ah05g', 'ah05h', 'ah05i', 'ah05n', 'ah06a_1', 'ah06a_2', 'ah06b_1', 'ah06b_2', 'ah06g_1', 'ah06g_2', 'ah06h_1', 'ah06h_2', 'ah06i_1', '

{'meaning': 'HHM owns living house?',
 'values_label': {'Yes': '1', 'No': '3'},
 'missing_values': '47'}

In [8]:
# spot-check a few more columns
for key in ["folio", "ah03a", "ah04a_1", "ah04a_2"]:
    print(key, "->", enriched_with_missing[key])


folio -> {'meaning': 'Household ID', 'missing_values': '0'}
ah03a -> {'meaning': 'HHM owns living house?', 'values_label': {'Yes': '1', 'No': '3'}, 'missing_values': '47'}
ah04a_1 -> {'meaning': 'Value living house?', 'values_label': {'Yes': '1', 'DK': '8'}, 'missing_values': '3899'}
ah04a_2 -> {'meaning': 'Value living house', 'missing_values': '3899'}


In [ ]:
# ii_ah has no DK/NA-labelled columns, so try a table that does, to see the
# broadened definition actually change the number (not just fall back to NaN)
preview = add_missing_value_counts("ii_vlh", verbose=False)
for key in ["vlh05", "vlh06", "vlh16a"]:
    print(key, "->", preview[key])


## Step 3: write the new file (original `ii_ah_enriched.json` is left untouched)

In [9]:
def write_enriched_with_missing(table_name: str, enriched_dict: dict) -> Path:
    out_path = JSON_DIR / f"{table_name}_enriched_with_missing_values.json"

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(enriched_dict, f, indent=4, ensure_ascii=False)

    return out_path


out_path = write_enriched_with_missing(table_name, enriched_with_missing)
print("Wrote:", out_path.relative_to(ROOT))


Wrote: raw_data/hh09dta_b2/codebook_json/ii_ah_enriched_with_missing_values.json


## Step 3b: confirm the original file on disk was not touched

In [10]:
fresh_original = json.load(open(JSON_DIR / f"{table_name}_enriched.json", encoding="utf-8"))
fresh_new = json.load(open(out_path, encoding="utf-8"))

assert "missing_values" not in fresh_original["ah03a"], "original file was modified on disk!"
assert "missing_values" in fresh_new["ah03a"], "new file is missing the field"

print("Original ii_ah_enriched.json untouched. New file ii_ah_enriched_with_missing_values.json is correct.")
print()
print("original:", fresh_original["ah03a"])
print("new:     ", fresh_new["ah03a"])


Original ii_ah_enriched.json untouched. New file ii_ah_enriched_with_missing_values.json is correct.

original: {'meaning': 'HHM owns living house?', 'values_label': {'Yes': '1', 'No': '3'}}
new:      {'meaning': 'HHM owns living house?', 'values_label': {'Yes': '1', 'No': '3'}, 'missing_values': '47'}


## Step 4 (optional): do the same for every table

Only run this once you're happy with `ii_ah`'s output above. This will create one
`{table}_enriched_with_missing_values.json` per existing `{table}_enriched.json` in `codebook_json/`.

In [11]:
all_tables = sorted(
    p.stem.replace("_enriched", "")
    for p in JSON_DIR.glob("*_enriched.json")
)
print(f"{len(all_tables)} tables found:", all_tables)


14 tables found: ['ii_ah', 'ii_conpor', 'ii_crh', 'ii_in', 'ii_inr', 'ii_ne', 'ii_nna', 'ii_nna1', 'ii_portad', 'ii_se', 'ii_su', 'ii_su1', 'ii_vlh', 'ii_vlh1']


In [12]:
for t in all_tables:
    enriched = add_missing_value_counts(t, verbose=False)
    written_path = write_enriched_with_missing(t, enriched)
    print(f"[{t}] wrote {written_path.name}")


[ii_ah] in dataframe but not in codebook (no missing_values added): ['ah05a', 'ah05b', 'ah05g', 'ah05h', 'ah05i', 'ah05n', 'ah06a_1', 'ah06a_2', 'ah06b_1', 'ah06b_2', 'ah06g_1', 'ah06g_2', 'ah06h_1', 'ah06h_2', 'ah06i_1', 'ah06i_2', 'ah06n_1', 'ah06n_2']
[ii_ah] wrote ii_ah_enriched_with_missing_values.json
[ii_conpor] wrote ii_conpor_enriched_with_missing_values.json
[ii_crh] wrote ii_crh_enriched_with_missing_values.json
[ii_in] wrote ii_in_enriched_with_missing_values.json
[ii_inr] wrote ii_inr_enriched_with_missing_values.json
[ii_ne] wrote ii_ne_enriched_with_missing_values.json
[ii_nna] wrote ii_nna_enriched_with_missing_values.json
[ii_nna1] wrote ii_nna1_enriched_with_missing_values.json
[ii_portad] wrote ii_portad_enriched_with_missing_values.json
[ii_se] wrote ii_se_enriched_with_missing_values.json
[ii_su] wrote ii_su_enriched_with_missing_values.json
[ii_su1] wrote ii_su1_enriched_with_missing_values.json
[ii_vlh] wrote ii_vlh_enriched_with_missing_values.json
[ii_vlh1] wro